# Experimento 1: recuperación densa, híbrida y con reranking

**Pregunta experimental.** Sobre el mismo benchmark de artículos reales del Código Civil, ¿cómo cambian la cobertura (Recall@k), el orden (MRR, nDCG@5) y la latencia cuando se agrega BM25 con fusión RRF a la recuperación densa, y cuando luego se agrega un Cross-Encoder?

```text
A = denso: E5 + FAISS IndexFlatIP
B = A + BM25 + RRF
C = B + Cross-Encoder
```

A frente a B cambia dos cosas a la vez (señal léxica y fusión), así que una diferencia no se puede atribuir a una sola. B frente a C agrega solo el reranker: el Cross-Encoder reordena los 20 candidatos de B y no puede traer evidencia que B no haya recuperado.

Fijos en las tres condiciones: corpus, 20 consultas, qrels, unidad de recuperación (un artículo), modelos, `candidate_depth = 20`, constante RRF = 60, semilla.

## 0. Entorno

In [1]:
!python -m pip install -q --break-system-packages -r ../requirements.txt

In [2]:
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

import config, corpus, metricas
print("raíz:", ROOT)
print("RUN_REAL_RETRIEVAL:", config.RUN_REAL_RETRIEVAL, "| RUN_REAL_RERANKER:", config.RUN_REAL_RERANKER,
      "| RUN_REAL_LLM:", config.RUN_REAL_LLM)

raíz: /workspace/CC-0F4/Evaluaciones/parcial
RUN_REAL_RETRIEVAL: True | RUN_REAL_RERANKER: True | RUN_REAL_LLM: True


## 1. Corpus, consultas y qrels

Corpus: artículos vigentes del Código Civil peruano (D.L. 295), versión `corpus_cc_peru_v2.json` del proyecto de tesis. Cada artículo es un passage con identificador estable `cc_art_N`. Se excluyen los 20 artículos derogados.

Las 20 consultas y sus qrels se anotaron antes de ejecutar cualquier recuperador, leyendo el texto de cada artículo. Ningún qrel usa los 7 artículos marcados para verificar en el SPIJ.

In [3]:
passages = corpus.load_passages()
queries, qrels, references = corpus.load_benchmark()
stats = corpus.validate_benchmark(passages, queries, qrels, references)
print(stats)

words = pd.Series([p["num_palabras"] for p in passages])
print(f"palabras por artículo: media {words.mean():.1f}, mediana {words.median():.0f}, máximo {words.max()}")
display(pd.Series([p["libro"] for p in passages]).value_counts().rename("artículos vigentes").to_frame())

{'passages': 2115, 'consultas': 20, 'pares_relevantes': 31, 'consultas_rag': 10, 'controles_negativos': 2}
palabras por artículo: media 43.4, mediana 33, máximo 500


,artículos vigentes
Fuentes de las Obligaciones,620
Derecho de Familia,437
Derechos Reales,251
Derecho de Sucesiones,221
Las Obligaciones,219
Derecho de las Personas,139
Acto Jurídico,94
Derecho Internacional Privado,65
Registros Públicos,39
Prescripción y Caducidad,19


In [4]:
q_df = pd.DataFrame([{"query_id": q["query_id"], "tipo": q["tipo"], "consulta": q["query"],
                      "qrels": ", ".join(qrels[q["query_id"]])} for q in queries])
display(q_df)
print(q_df.tipo.value_counts().to_dict())

,query_id,tipo,consulta,qrels
0,C01,directa,¿A partir de qué edad una persona tiene plena capacidad de ejercicio?,cc_art_42
1,C02,contradiccion,¿Desde qué edad el matrimonio hace cesar la incapacidad de un menor?,"cc_art_42, cc_art_46"
2,C03,directa,¿En qué plazo puede un asociado impugnar judicialmente un acuerdo de la asociación?,cc_art_92
3,C04,directa,¿Cómo se computa un plazo señalado por días según el Código Civil?,"cc_art_183, cc_art_184"
4,C05,contradiccion,"Si el último día de un plazo de caducidad cae en día inhábil, ¿el plazo se extiende al siguiente día hábil?","cc_art_2007, cc_art_183"
5,C06,directa,¿En cuántos años prescribe la acción de nulidad del acto jurídico?,cc_art_2001
6,C07,parafraseada,"Firmé un contrato que puede anularse, ¿cuánto tiempo tengo para demandar su anulabilidad?",cc_art_2001
7,C08,directa,¿La caducidad admite interrupción o suspensión?,cc_art_2005
8,C09,directa,¿Desde cuándo empieza a correr el plazo de prescripción?,cc_art_1993
9,C10,parafraseada,"Vivo en una casa desde hace diez años como si fuera mía, ¿puedo convertirme en propietario?",cc_art_950


{'directa': 12, 'parafraseada': 5, 'contradiccion': 3}


## 2. Índices

Unidad de recuperación: un artículo por chunk. En la PC2 el tamaño de fragmento no cambió Recall@3, y el artículo es la unidad que se cita en derecho, así que la cita del RAG apunta directamente a un `passage_id`. Con un artículo por chunk el ranking de chunks coincide con el ranking de passages que usan los qrels y no hace falta deduplicar solapamientos.

E5 lee como máximo 512 tokens. La celda cuenta cuántos artículos pasan ese límite y si alguno es relevante para una consulta, porque un artículo truncado se indexa sin su parte final.

In [5]:
from recuperacion import BM25, DenseRetriever, Reranker, run_condition, CONDITIONS

t0 = time.perf_counter()
dense = DenseRetriever(passages)
bm25 = BM25(passages)
reranker = Reranker()
print(f"índices construidos en {time.perf_counter() - t0:.1f} s")
print("denso:", dense.implementation)
print("reranker:", reranker.implementation)

if dense.token_lengths is not None:
    tl = pd.Series(dense.token_lengths, index=[p["passage_id"] for p in passages])
    gold = {pid for v in qrels.values() for pid in v}
    over = tl[tl > dense.max_seq_length]
    print(f"tokens por artículo: media {tl.mean():.0f}, máximo {tl.max()} (límite {dense.max_seq_length})")
    print(f"artículos truncados: {len(over)} -> {list(over.index)}")
    print(f"artículos relevantes truncados: {sorted(gold & set(over.index))}")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 227.32it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (526 > 512). Running this sequence through the model will result in indexing errors
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 707.13it/s]


índices construidos en 146.8 s
denso: intfloat/multilingual-e5-small + FAISS IndexFlatIP
reranker: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
tokens por artículo: media 69, máximo 692 (límite 512)
artículos truncados: 4 -> ['cc_art_104', 'cc_art_274', 'cc_art_277', 'cc_art_1138']
artículos relevantes truncados: []


Comprobación de la Semana 4: con vectores normalizados, el score de `IndexFlatIP` es el coseno.

In [6]:
q = queries[0]["query"]
qv = dense.encode_query(q)[0]
top = dense.rank(q, 1)[0]
pv = dense.vectors[[p["passage_id"] for p in passages].index(top["passage_id"])]
cos = float(qv @ pv / (np.linalg.norm(qv) * np.linalg.norm(pv)))
print(f"{top['passage_id']}: score índice {top['score']:.4f} | coseno manual {cos:.4f} | normas {np.linalg.norm(qv):.4f}, {np.linalg.norm(pv):.4f}")
assert abs(top["score"] - cos) < 1e-4

cc_art_42: score índice 0.9346 | coseno manual 0.9346 | normas 1.0000, 1.0000


## 3. Ejecución de A, B y C

Antes de medir se hace una consulta de calentamiento que no se registra. La latencia excluye la carga de modelos y la construcción de índices; incluye la codificación de la consulta, BM25, RRF y el reranker. Las tres etapas corren en secuencia en CPU.

In [7]:
for c in CONDITIONS:
    run_condition(c, "consulta de calentamiento", dense, bm25, reranker)

rows, rankings = [], []
for cond in CONDITIONS:
    for q in queries:
        qid = q["query_id"]
        ranking, timing = run_condition(cond, q["query"], dense, bm25, reranker)
        ids = [r["passage_id"] for r in ranking]
        gold = qrels[qid]
        row = {"condition": cond, "query_id": qid, "tipo": q["tipo"], **{k: round(v, 2) for k, v in timing.items()}}
        for k in config.TOP_K_VALUES:
            row[f"recall@{k}"] = metricas.recall_at_k(ids, gold, k)
        row["mrr"] = metricas.reciprocal_rank(ids, gold)
        row["ndcg@5"] = metricas.ndcg_at_k(ids, gold, config.NDCG_K)
        row["rango_primer_relevante"] = metricas.first_relevant_rank(ids, gold)
        row["fallo"] = metricas.classify_failure(ids, gold)
        rows.append(row)
        rankings.append({"condition": cond, "query_id": qid,
                         "ranking": [[r["passage_id"], round(r["score"], 5)] for r in ranking]})

per_query = pd.DataFrame(rows)
per_query.to_csv("results/exp1_por_consulta.csv", index=False)
with open("results/exp1_rankings.jsonl", "w", encoding="utf-8") as fh:
    for r in rankings:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")
print("filas:", len(per_query))

filas: 60


## 4. Resultados agregados

In [8]:
def p95(s):
    return float(np.percentile(s, 95))

metric_cols = ["recall@1", "recall@3", "recall@5", "mrr", "ndcg@5"]
summary = per_query.groupby("condition")[metric_cols].mean().round(3)
lat = per_query.groupby("condition").agg(
    dense_ms=("dense_ms", "mean"), bm25_ms=("bm25_ms", "mean"), rrf_ms=("rrf_ms", "mean"),
    reranker_ms=("reranker_ms", "mean"), total_ms=("total_ms", "mean"), total_p95_ms=("total_ms", p95)).round(1)
summary = summary.join(lat)
summary.insert(0, "sistema", [CONDITIONS[c] for c in summary.index])
summary.to_csv("results/exp1_resumen.csv")
summary

,sistema,recall@1,recall@3,recall@5,mrr,ndcg@5,dense_ms,bm25_ms,rrf_ms,reranker_ms,total_ms,total_p95_ms
condition,,,,,,,,,,,,
A,denso,0.442,0.658,0.700,0.688,0.642,60.5,0.0,0.0,0.0,60.5,94.6
B,denso + BM25 + RRF,0.392,0.558,0.650,0.629,0.582,48.7,3.3,0.1,0.0,52.1,74.0
C,denso + BM25 + RRF + Cross-Encoder,0.542,0.633,0.675,0.750,0.666,58.0,4.2,0.1,3061.1,3123.4,4958.3


In [9]:
by_type = per_query.groupby(["tipo", "condition"])[["recall@3", "mrr", "ndcg@5"]].mean().round(3).unstack("condition")
by_type.to_csv("results/exp1_por_tipo.csv")
by_type

recall@3                  mrr               ndcg@5              
condition            A      B      C      A      B      C      A      B      C
tipo                                                                          
contradiccion    0.667  0.500  0.667  0.833  0.733  0.833  0.667  0.617  0.667
directa          0.764  0.722  0.722  0.814  0.766  0.875  0.767  0.700  0.776
parafraseada     0.400  0.200  0.400  0.300  0.240  0.400  0.326  0.277  0.400

## 5. Incertidumbre: bootstrap pareado

Las mismas 20 consultas pasan por A, B y C, así que se remuestrean consultas completas (5000 veces) y se calcula el intervalo de la diferencia media. Un intervalo que cruza cero significa que estos datos son compatibles con mejora y con empeoramiento.

In [10]:
boot = pd.DataFrame([metricas.paired_bootstrap(per_query, m, a, b)
                     for m in ["recall@3", "mrr", "ndcg@5"] for a, b in [("A", "B"), ("B", "C")]])
boot.to_csv("results/exp1_bootstrap.csv", index=False)
boot

,metrica,comparacion,delta_media,ic95_inf,ic95_sup,ic_cruza_cero,consultas_mejoran,consultas_empeoran
0,recall@3,B - A,-0.1000,-0.2250,0.0000,True,0,3
1,recall@3,C - B,0.0750,-0.0250,0.2250,True,3,1
2,mrr,B - A,-0.0589,-0.1745,0.0472,True,3,4
3,mrr,C - B,0.1206,-0.0072,0.2612,True,6,1
4,ndcg@5,B - A,-0.0601,-0.1305,0.0085,True,2,5
5,ndcg@5,C - B,0.0842,0.0010,0.1789,False,5,2


## 6. Tipos de fallo (Cuaderno 7)

In [11]:
fail = pd.crosstab(per_query.condition, per_query.fallo)
fail.to_csv("results/exp1_fallos.csv")
fail

fallo,PARTIAL_RECALL,RANKING_ERROR,RETRIEVAL_MISS,RETRIEVAL_OK
condition,,,,
A,6,1,3,10
B,4,4,3,9
C,7,1,3,9


## 7. Consultas con diferencias entre condiciones

`detalle(query_id)` muestra el top-5 de cada condición, marca los artículos relevantes e indica el puesto de cada qrel dentro de los 20 candidatos.

In [12]:
pivot = per_query.pivot(index="query_id", columns="condition", values=["recall@3", "mrr", "ndcg@5"])
deltas = pd.DataFrame({
    "tipo": per_query[per_query.condition == "A"].set_index("query_id").tipo,
    "mrr_A": pivot[("mrr", "A")].round(3), "mrr_B": pivot[("mrr", "B")].round(3), "mrr_C": pivot[("mrr", "C")].round(3),
    "R3_A": pivot[("recall@3", "A")], "R3_B": pivot[("recall@3", "B")], "R3_C": pivot[("recall@3", "C")],
})
deltas["cambia"] = (deltas[["mrr_A", "mrr_B", "mrr_C"]].nunique(axis=1) > 1) | (deltas[["R3_A", "R3_B", "R3_C"]].nunique(axis=1) > 1)
deltas.to_csv("results/exp1_deltas.csv")
deltas

,tipo,mrr_A,mrr_B,mrr_C,R3_A,R3_B,R3_C,cambia
query_id,,,,,,,,
C01,directa,1.000,1.000,1.000,1.000000,1.000000,1.000000,False
C02,contradiccion,1.000,0.200,1.000,0.500000,0.000000,0.500000,True
C03,directa,1.000,1.000,1.000,1.000000,1.000000,1.000000,False
C04,directa,1.000,1.000,1.000,0.500000,0.500000,0.500000,False
C05,contradiccion,1.000,1.000,1.000,1.000000,1.000000,1.000000,False
C06,directa,1.000,1.000,1.000,1.000000,1.000000,1.000000,False
C07,parafraseada,0.000,0.000,0.000,0.000000,0.000000,0.000000,False
C08,directa,1.000,1.000,1.000,1.000000,1.000000,1.000000,False
C09,directa,0.333,0.500,1.000,1.000000,1.000000,1.000000,True


In [13]:
ranking_by = {(r["condition"], r["query_id"]): r["ranking"] for r in rankings}
text_by = {p["passage_id"]: p["text"] for p in passages}

def detalle(qid, k=5):
    q = next(x for x in queries if x["query_id"] == qid)
    gold = qrels[qid]
    print(f"{qid} ({q['tipo']}): {q['query']}")
    print("qrels:", gold)
    for cond in CONDITIONS:
        ids = [pid for pid, _ in ranking_by[(cond, qid)]]
        pos = {g: (ids.index(g) + 1 if g in ids else None) for g in gold}
        print(f"\n  {cond} ({CONDITIONS[cond]}) | puesto de cada qrel en los 20 candidatos: {pos}")
        for i, (pid, score) in enumerate(ranking_by[(cond, qid)][:k], start=1):
            mark = "RELEVANTE" if pid in gold else ""
            print(f"    {i}. {pid:14s} {score:9.4f} {mark:9s} {text_by[pid][:90].replace(chr(10), ' ')}")

for qid in deltas[deltas.cambia].index[:3]:
    detalle(qid)
    print()

C02 (contradiccion): ¿Desde qué edad el matrimonio hace cesar la incapacidad de un menor?
qrels: ['cc_art_42', 'cc_art_46']

  A (denso) | puesto de cada qrel en los 20 candidatos: {'cc_art_42': None, 'cc_art_46': 1}
    1. cc_art_46         0.8860 RELEVANTE Artículo 46. Capacidad adquirida por matrimonio o título oficial. La incapacidad de las pe
    2. cc_art_549        0.8782           Artículo 549. La tutela se acaba: 1.- Por la muerte del menor. 2.- Por llegar el menor a l
    3. cc_art_537        0.8753           Artículo 537. La acción del menor para anular los actos celebrados por el tutor sin las fo
    4. cc_art_401        0.8729           Artículo 401. El hijo menor o incapaz puede en todo caso negar el reconocimiento hecho en 
    5. cc_art_368        0.8694           Artículo 368. La acción puede ser ejercida por los ascendientes del marido, en los casos d

  B (denso + BM25 + RRF) | puesto de cada qrel en los 20 candidatos: {'cc_art_42': None, 'cc_art_46': 5}
    1. cc_ar

## 8. Manifiesto de la corrida

In [14]:
import platform
manifest = {
    "fecha_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "python": platform.python_version(),
    "modos": {"retrieval_real": config.RUN_REAL_RETRIEVAL, "reranker_real": config.RUN_REAL_RERANKER},
    "modelos": {"denso": dense.implementation, "reranker": reranker.implementation},
    "parametros": {"candidate_depth": config.CANDIDATE_DEPTH, "rrf_constant": config.RRF_CONSTANT,
                   "bm25_k1": config.BM25_K1, "bm25_b": config.BM25_B, "semilla": config.SEED,
                   "bootstrap": config.BOOTSTRAP_RESAMPLES},
    "datos": {"passages": len(passages), "consultas": len(queries),
              "sha256": {p.name: corpus.sha256_file(p) for p in [config.CORPUS_PATH, config.QUERIES_PATH, config.QRELS_PATH]}},
}
Path("results/exp1_manifiesto.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
if not (config.RUN_REAL_RETRIEVAL and config.RUN_REAL_RERANKER):
    print("Corrida de validación del software: estos números no se reportan como resultados.")

{
  "fecha_utc": "2026-10-29T08:07:03Z",
  "python": "3.12.3",
  "modos": {
    "retrieval_real": true,
    "reranker_real": true
  },
  "modelos": {
    "denso": "intfloat/multilingual-e5-small + FAISS IndexFlatIP",
    "reranker": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
  },
  "parametros": {
    "candidate_depth": 20,
    "rrf_constant": 60,
    "bm25_k1": 1.5,
    "bm25_b": 0.75,
    "semilla": 42,
    "bootstrap": 5000
  },
  "datos": {
    "passages": 2115,
    "consultas": 20,
    "sha256": {
      "corpus_cc_peru_v2.json": "1fd2de29146dfd400b35215d04f6abe10e1a1541a7264e64eb20b4a8105974c4",
      "consultas.jsonl": "4b9bc35e0a8587ea1a57d71c44edeb3cb43a17940e167c0c9034aa28eefc0d27",
      "qrels.json": "c9986bef7ce95a23d26602d66af73875d0b91b1fd63e21c530b81c2ac7c98d93"
    }
  }
}
